In [0]:
bronze_df = spark.table(
    "ecommerce_project.bronze_orders"
)

display(bronze_df)

order_id,customer_id,product_id,amount,event_time
1001,101,501,500,2026-06-23 10:00:00
1001,101,501,550,2026-06-23 10:05:00
1001,101,501,600,2026-06-23 10:10:00
1002,102,502,700,2026-06-23 10:15:00
1003,null,503,900,2026-06-23 10:20:00


In [0]:
from pyspark.sql.functions import col
rejected_df = bronze_df.filter(
    col("customer_id").isNull()
)

display(rejected_df)

order_id,customer_id,product_id,amount,event_time
1003,null,503,900,2026-06-23 10:20:00


In [0]:
valid_df = bronze_df.filter(
    col("customer_id").isNotNull()
)

display(valid_df)

order_id,customer_id,product_id,amount,event_time
1001,101,501,500,2026-06-23 10:00:00
1001,101,501,550,2026-06-23 10:05:00
1001,101,501,600,2026-06-23 10:10:00
1002,102,502,700,2026-06-23 10:15:00


In [0]:
from pyspark.sql.functions import current_timestamp

silver_df = (bronze_df.filter(col("customer_id").isNotNull())
    .dropDuplicates(["order_id"])
    .withColumn("processed_timestamp", current_timestamp() ))

display(silver_df)

order_id,customer_id,product_id,amount,event_time,processed_timestamp
1001,101,501,500,2026-06-23 10:00:00,2026-07-05T14:11:15.999Z
1002,102,502,700,2026-06-23 10:15:00,2026-07-05T14:11:15.999Z


In [0]:
from pyspark.sql.functions import current_timestamp

silver_df = (
    bronze_df
        .dropDuplicates(["order_id"])
        .filter("customer_id IS NOT NULL")
        .withColumn(
            "processed_timestamp",
            current_timestamp()
        )
)

display(silver_df)

order_id,customer_id,product_id,amount,event_time,processed_timestamp
1001,101,501,500,2026-06-23 10:00:00,2026-07-05T14:11:17.439Z
1002,102,502,700,2026-06-23 10:15:00,2026-07-05T14:11:17.439Z


In [0]:
from pyspark.sql.functions import when

df = bronze_df.withColumn(
    "customer_validation",
    when(col("customer_id").isNull(),
         "null_customer"
         ).otherwise("PASS")
)

In [0]:
display(df)

order_id,customer_id,product_id,amount,event_time,customer_validation
1001,101,501,500,2026-06-23 10:00:00,PASS
1001,101,501,550,2026-06-23 10:05:00,PASS
1001,101,501,600,2026-06-23 10:10:00,PASS
1002,102,502,700,2026-06-23 10:15:00,PASS
1003,null,503,900,2026-06-23 10:20:00,null_customer


In [0]:
PASS = "PASS"
NULL_CUSTOMER = "NULL_CUSTOMER"
NEGATIVE_AMOUNT = "NEGATIVE_AMOUNT"

validated_df = bronze_df.withColumn(
    "customer_validation",
    when(col("customer_id").isNull(),
         NULL_CUSTOMER
         ).otherwise(PASS)
).withColumn(
    "amount_validation",
    when(col("amount") < 0,
         NEGATIVE_AMOUNT
         ).otherwise(PASS)
)

In [0]:
display(validated_df)

order_id,customer_id,product_id,amount,event_time,customer_validation,amount_validation
1001,101,501,500,2026-06-23 10:00:00,PASS,PASS
1001,101,501,550,2026-06-23 10:05:00,PASS,PASS
1001,101,501,600,2026-06-23 10:10:00,PASS,PASS
1002,102,502,700,2026-06-23 10:15:00,PASS,PASS
1003,null,503,900,2026-06-23 10:20:00,NULL_CUSTOMER,PASS


In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, row_number

window_spec = Window.partitionBy("order_id").orderBy(col("event_time").desc())

ranked_df = bronze_df.withColumn(
    "row_number",
    row_number().over(window_spec)
)

In [0]:
display(ranked_df)

order_id,customer_id,product_id,amount,event_time,row_number
1001,101,501,600,2026-06-23 10:10:00,1
1001,101,501,550,2026-06-23 10:05:00,2
1001,101,501,500,2026-06-23 10:00:00,3
1002,102,502,700,2026-06-23 10:15:00,1
1003,null,503,900,2026-06-23 10:20:00,1


In [0]:
LATEST_RECORD = 1

silver_df = (
    ranked_df
        .filter(col("row_number") == LATEST_RECORD)
        .drop("row_number")
        .withColumn("processed_timestamp", current_timestamp())
)

display(silver_df)

order_id,customer_id,product_id,amount,event_time,processed_timestamp
1001,101,501,600,2026-06-23 10:10:00,2026-07-05T14:11:23.949Z
1002,102,502,700,2026-06-23 10:15:00,2026-07-05T14:11:23.949Z
1003,null,503,900,2026-06-23 10:20:00,2026-07-05T14:11:23.949Z


In [0]:
display(silver_df)

order_id,customer_id,product_id,amount,event_time,processed_timestamp
1001,101,501,600,2026-06-23 10:10:00,2026-07-05T14:11:25.000Z
1002,102,502,700,2026-06-23 10:15:00,2026-07-05T14:11:25.000Z
1003,null,503,900,2026-06-23 10:20:00,2026-07-05T14:11:25.000Z


In [0]:
silver_df.write \
    .format("delta") \
        .mode("overwrite") \
            .saveAsTable("ecommerce_project.silver_orders")